# OrganoidEnv — Multi-Seed Training on Google Colab

Runs the full multi-seed training + ablation pipeline for **OrganoidEnv**.

**Click Runtime → Run All.**

In [ ]:
# 1. Clean installation of dependencies (NumPy 2.x + Brian2 >= 2.8 compatible)
import sys
print(f'Python version: {sys.version}')

# Uninstall stale packages that might hold older incompatible Brian2 versions
!pip uninstall -y -q brian2genn brian2
!pip install -q --upgrade pip
!pip install -q "brian2>=2.8.0" "numpy>=2.0.0" gymnasium torch matplotlib pandas scipy

In [ ]:
# 2. Clone the repository and install
import os
if os.path.exists('/content/Organoid-Intelligence-gym-'):
    %cd /content/Organoid-Intelligence-gym-
    !git pull
else:
    %cd /content
    !git clone https://github.com/vansh7nvc/Organoid-Intelligence-gym-.git
    %cd Organoid-Intelligence-gym-
!pip install -q -e .

In [ ]:
# 3. Verify Setup
import numpy as np
import brian2 as b2
import torch
import gymnasium

print(f'NumPy version:      {np.__version__}')
print(f'Brian2 version:     {b2.__version__}')
print(f'Brian2 backend:     {b2.prefs.codegen.target}')
print(f'PyTorch version:    {torch.__version__}')
print(f'CUDA available:     {torch.cuda.is_available()}')
print(f'Gymnasium version:  {gymnasium.__version__}')
print('\n✅ All imports OK!')

In [ ]:
# 4. Quick Smoke Test (5 simulation steps)
import sys
sys.path.insert(0, '.')
from organoid_rl.environment.core import OrganoidEnv
from organoid_rl.agents.dqn_agent import DQNAgent

env = OrganoidEnv()
agent = DQNAgent(obs_dim=21, n_actions=8)
state, info = env.reset()
for step in range(5):
    action = agent.choose_action(state)
    state, reward, term, trunc, info = env.step(action)
    agent.store_transition(state, action, reward, state, term or trunc)
    if term or trunc:
        break
print(f'Smoke test passed: state.shape={state.shape}, reward={reward:.2f}')
print('✅ Environment + Agent working!')

In [ ]:
# 5. Mount Google Drive (for checkpoint persistence)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted.')
except:
    print('⚠️ Drive not available. Results saved locally.')

In [ ]:
# 6. Run Multi-Seed Training (3 seeds × 200 episodes)
# Checkpoints saved every 25 episodes to Google Drive.
# If disconnected, simply re-run this cell to resume automatically.
!PYTHONPATH="." python organoid_rl/experiments/publication/20_colab_multiseed_v2.py

In [ ]:
# 7. Run Ablation Experiments (No_GAR + No_SDM, 3 seeds × 100 episodes each)
!PYTHONPATH="." python organoid_rl/experiments/publication/21_colab_ablations_v2.py

In [ ]:
# 8. Plot Results
import pandas as pd
import matplotlib.pyplot as plt
import glob

res_dir = 'organoid_rl/experiments/publication/colab_results_v2'

# --- Multi-seed learning curves ---
seed_files = sorted(glob.glob(f'{res_dir}/training_log_seed_*.csv'))
if seed_files:
    fig, ax = plt.subplots(figsize=(10, 5))
    for f in seed_files:
        df = pd.read_csv(f)
        seed_val = int(df['seed'].iloc[0])
        ax.plot(df['episode'], df['reward'].rolling(10).mean(), alpha=0.6, label=f'Seed {seed_val}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward (10-ep rolling avg)')
    ax.set_title('OrganoidEnv: Multi-Seed Training')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{res_dir}/multiseed_learning_curve.png', dpi=150)
    plt.show()
else:
    print('No multi-seed results found yet.')

# --- Ablation comparison ---
abl_files = sorted(glob.glob(f'{res_dir}/training_log_*_seed_*.csv'))
if abl_files:
    fig, ax = plt.subplots(figsize=(10, 5))
    all_abl = pd.concat([pd.read_csv(f) for f in abl_files])
    for name, grp in all_abl.groupby('ablation'):
        mean = grp.groupby('episode')['reward'].mean()
        ax.plot(mean.index, mean.rolling(10).mean(), label=name)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward (10-ep rolling avg)')
    ax.set_title('Ablation Study')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{res_dir}/ablation_comparison.png', dpi=150)
    plt.show()
else:
    print('No ablation results found yet.')

print('\n✅ Done! Plots saved to:', res_dir)